In [0]:
%pip install geopandas rasterio shapely

In [0]:
import os
import glob
import geopandas as gpd
import rasterio
from rasterio.mask import mask
from shapely.geometry import mapping
import tempfile
import shutil

In [0]:
# ── Get the list of pending flights from the orchestrator ──────────────────
# This task only runs when check_pending_location_clips routed us here (i.e.
# "proceed" == "true"). We read the actual list of flight paths that
# 3_orquestator identified as still needing location clipping.
flight_list = dbutils.jobs.taskValues.get(
    taskKey="3_orquestator",
    key="missing_clips",
    default=[]
)
 
# Safety net: if for some reason the list is empty, stop here instead of
# looping over nothing.
if not flight_list:
    dbutils.notebook.exit("No pending flights to process. Exiting gracefully.")
 
print(f" Received {len(flight_list)} flights for plot cropping.\n")
 
# ── Process each pending flight one at a time ───────────────────────────────
for flight_path in flight_list:
    print("-" * 60)
    print(f" PROCESSING FLIGHT: {flight_path}")
 
    # DATABRICKS FIX: convert "dbfs:/" paths to "/dbfs/" so standard Python
    # file operations (os.path, open, etc.) can read them correctly.
    if flight_path.startswith("dbfs:/"):
        flight_path = flight_path.replace("dbfs:/", "/dbfs/")
 
    # The flight's own folder, and its parent folder (used to find the
    # separate "field_data" folder that holds the boundary geometry).
    base_dir = os.path.dirname(flight_path)
    parent_dir = os.path.dirname(base_dir)
    field_data_dir = os.path.join(parent_dir, "field_data")
 
    # ── Locate the orthomosaic file for this flight ─────────────────────────
    # Same logic as in 2_5_Update_ortho_table: check for either RGB.tif or
    # MS.tif, since flights can be either sensor type.
    possible_ortho_names = ["RGB.tif", "MS.tif"]
    rgb_path = None
    for name in possible_ortho_names:
        candidate = os.path.join(base_dir, name)
        if os.path.exists(candidate):
            rgb_path = candidate
            break
 
    # Clipped output will be saved back into the flight's own folder.
    out_dir = os.path.join(base_dir)
 
    print(f" Searching geometries in: {field_data_dir}")
    # The boundary geometry that defines the area to crop out of the
    # orthomosaic (a GeoJSON file with the location's boundary polygon).
    vector_path = os.path.join(field_data_dir, "location_boundary.geojson")
 
    # If there's no orthomosaic yet, we can't clip anything — skip this flight.
    if rgb_path is None:
        print(f" Error: No RGB.tif or MS.tif found in {base_dir}. Skipping flight...")
        continue
    else:
        ortho_name = os.path.splitext(os.path.basename(rgb_path))[0]
        print(f" Orthomosaic found: {os.path.basename(rgb_path)}")
 
    # If there's no boundary file, we also can't clip anything — skip this flight.
    if not os.path.exists(vector_path):
       print(f" Error: No 'location_boundary.geojson' found in {field_data_dir}. Skipping flight...")
       continue
 
    print(f" Vector file found: {os.path.basename(vector_path)}")
 
    os.makedirs(out_dir, exist_ok=True)
 
    # Local scratch directory on the cluster's own disk (not the Volume).
    # Writing/reading raster files directly on cloud storage (the Volume)
    # can cause seek/IO errors, so we work locally first and copy results
    # over at the end.
    local_tmp_dir = tempfile.mkdtemp(prefix="plot_clip_")
 
    try:
        print(" Loading geometries and checking Coordinate Reference Systems (CRS)...")
        # Load the boundary polygon(s) from the GeoJSON file.
        gdf_plots = gpd.read_file(vector_path)
 
        with rasterio.open(rgb_path) as src:
            raster_crs = src.crs
 
            # The boundary polygons and the orthomosaic must be in the same
            # Coordinate Reference System (CRS) for the crop to align
            # correctly. If they don't match, reproject the polygons to
            # match the raster's CRS.
            if gdf_plots.crs != raster_crs:
                print(f" Reprojecting polygons from {gdf_plots.crs} to {raster_crs}...")
                gdf_plots = gdf_plots.to_crs(raster_crs)
 
            # Choose the output format based on how many bands the
            # orthomosaic has: standard RGB/RGBA (<=4 bands) can be saved as
            # PNG, while multispectral imagery (more bands) needs GeoTIFF to
            # preserve all the band data.
            n_bands = src.count
            if n_bands <= 4:
                out_driver = "PNG"
                out_ext = "png"
            else:
                out_driver = "GTiff"
                out_ext = "tif"
            print(f" Detected {n_bands} bands -> using {out_driver} for output ({out_ext}).")
 
            print(f" Clipping {len(gdf_plots)} detected plots...")
 
            # ── Crop the orthomosaic to each boundary polygon ───────────────
            for idx, row in gdf_plots.iterrows():
                geometry = [mapping(row.geometry)]
 
                # Crop the raster to just the area covered by this polygon.
                out_image, out_transform = mask(src, geometry, crop=True)
 
                # Update the raster metadata to match the new cropped size
                # and output format.
                out_meta = src.meta.copy()
                out_meta.update({
                    "driver": out_driver,
                    "height": out_image.shape[1],
                    "width": out_image.shape[2],
                    "transform": out_transform
                })
 
                file_name = f"{ortho_name}.{out_ext}"
 
                # Write to LOCAL disk first (avoids seek errors on the Volume).
                local_out_path = os.path.join(local_tmp_dir, file_name)
                with rasterio.open(local_out_path, "w", **out_meta) as dest:
                    dest.write(out_image)
 
                # Then copy the finished file to the Volume (final destination).
                final_out_path = os.path.join(out_dir, file_name)
                shutil.copyfile(local_out_path, final_out_path)
 
            print(f" SUCCESS: {len(gdf_plots)} images saved to {out_dir}")
 
    except Exception as e:
        # If anything fails for this flight, log it and move on to the next
        # one rather than stopping the whole batch.
        print(f" An error occurred processing this flight: {e}")
 
    finally:
        # Clean up local scratch files regardless of success/failure.
        shutil.rmtree(local_tmp_dir, ignore_errors=True)
 
print("\n" + "="*70)
print(" LOCATION CROPPING PIPELINE FINISHED SUCCESSFULLY.")